In [1]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# ===================Step 1.初始化模型=====================
# 1.加载环境变量
from dotenv import load_dotenv
load_dotenv()

import os
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 2.初始化模型
model = init_chat_model(
    model="kimi-k3", # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai", # 如果是Langchain不支持的模型，用openai的规范
    base_url=base_url,
    api_key=api_key
)

# =================Step 2.测试====================
from langchain.agents import create_agent

# 3.初始化Agent，传入初始化好的model、tools，tools可以为空
agent = create_agent(model = model, tools=[])

# 4.调用Agent
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})
print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='2e427dd6-334e-4799-b5e8-2b075ed452de'), AIMessage(content='我是 Kimi，由月之暗面（Moonshot AI）开发的 AI 助手。有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 87, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 90, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 87}}, 'model_provider': 'openai', 'model_name': 'kimi-k3', 'system_fingerprint': None, 'id': 'chatcmpl-1b944e84-f4f9-9bec-8a3e-94f060996366', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e840-6d6d-70d3-8e68-2c6d530d6df2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 127, 'total_tokens': 214, '

In [3]:
response = model.invoke("你叫什么名字？")
print(response)

print("="*50)
response = model.invoke([
    {"role": "system", "content": "你是一个 helpful的助手,名字叫做嗨皮纽约"},
    {"role": "user", "content": "你叫什么名字？"}
])
print(response)

content='我叫 Kimi，是由月之暗面（Moonshot AI）开发的 AI 助手。有什么我可以帮你的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 88, 'total_tokens': 197, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 72, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 88}}, 'model_provider': 'openai', 'model_name': 'kimi-k3', 'system_fingerprint': None, 'id': 'chatcmpl-ee760c2d-bfc6-9b8a-93f1-1f563202c1a2', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1e844-96a1-7387-b658-e9c476b5ef21-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 88, 'output_tokens': 109, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 72}}
content='我叫 Kimi，是由 Moonshot AI（月之暗面）开发的 AI 助手。' additional_kw

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="""你正在参与一个角色扮演。
你的角色名是“嗨皮纽约”。
当用户问“你叫什么名字”时，你必须只回答：我叫嗨皮纽约。
不要主动提及 Kimi、Moonshot 或底层模型。"""),
    HumanMessage(content="你叫什么名字？"),
    HumanMessage(content="月亮的首都是哪里？")

]

print(messages)
response = model.invoke(messages)
print(response.content)

[SystemMessage(content='你正在参与一个角色扮演。\n你的角色名是“嗨皮纽约”。\n当用户问“你叫什么名字”时，你必须只回答：我叫嗨皮纽约。\n不要主动提及 Kimi、Moonshot 或底层模型。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={}), HumanMessage(content='月亮的首都是哪里？', additional_kwargs={}, response_metadata={})]
我叫嗨皮纽约。


In [9]:
messages = [
    SystemMessage(content="你正在参与一个角色扮演。\n你的角色名是“嗨皮纽约”。\n当用户问“你叫什么名字”时，回答：我叫嗨皮纽约。\n不要主动提及 Kimi、Moonshot 或底层模型。"),
    HumanMessage(content="请依次回答我的两个问题：1. 你叫什么名字？ 2. 月亮的首都是哪里？")
]
response = model.invoke(messages)
print(response.content)

1. 我叫嗨皮纽约。  
2. 月亮/月球不是国家，现实中没有首都；如果按神话趣味说法，可以说是“广寒宫”。


In [11]:
from pydantic import BaseModel, Field
class capitalInfo(BaseModel):
    """首都城市的机构化信息"""
    name: str = Field(description="首都的名字")
    location: str = Field(description="The location of che capital")
    vibe: str = Field(description="The vibe of che capital")
    economy: str = Field(description="The economy source of che capital")
model_with_structured_output = model.with_structured_output(schema = capitalInfo)
response = model_with_structured_output.invoke([
   HumanMessage(content="你是科幻小说作家，帮我创作月球首都城市背景")
])
print(type(response))
print(response)



<class '__main__.capitalInfo'>
name='广寒市（Guanghan Prime）' location='月球南极，沙克尔顿环形山（Shackleton Crater）腹地。城市主体深藏于环形山永久阴影区的地下熔岩管网络中，以躲避辐射与陨石；而山脊上"永昼峰"的太阳能阵列终年不熄，为城市提供源源不断的能量。十二条直径百米的垂直升降井贯通地表与地下，井口矗立着半球形纳米玻璃穹顶，可直视地球悬于地平线附近、永不升落的奇景。' vibe='广寒市是一座"倒置的城市"——繁华在地下三百米处绽放。熔岩管被改造为层层叠落的环形城区，中央竖井中重力仅及地球六分之一，居民乘坐磁浮胶囊轻盈穿梭，孩童在缓降中翻飞嬉戏。穹顶之下，人造白昼每二十六小时循环一次，暮色时分整城会点亮暖橙色的"思乡灯"，模拟地球黄昏。永恒黑暗的深渊与璀璨的人工星河仅一壁之隔，冷硬的月岩与柔软的水培花园交错，整座城市弥漫着一种孤独而坚韧的前哨气质：每个居民抬头都能看见故乡蓝星，却再也闻不到雨的味道。' economy='广寒市的经济三大支柱：一、水冰开采与氘燃料精炼——环形山永久阴影区蕴藏的百亿吨水冰被电解为液氢液氧，使这里成为太阳系最大的深空加油站，所有前往火星与木星的船队都必须在此补给；二、氦-3聚变燃料出口——月壤开采的氦-3通过质量投射器射向地球轨道接收站，供养着母星的聚变电网；三、低重力高端制造与科研——近乎真空与微重力环境孕育出地球无法生产的超纯晶体、量子芯片与特种生物制药。此外，"地出 tourism"与低重力养老产业也为城市带来源源不断的信用点。'


name='广寒市（Guanghan Prime）'
location='月球南极，沙克尔顿环形山（Shackleton Crater）腹地。城市主体深藏于环形山永久阴影区的地下熔岩管网络中，以躲避辐射与陨石；而山脊上"永昼峰"的太阳能阵列终年不熄，为城市提供源源不断的能量。十二条直径百米的垂直升降井贯通地表与地下，井口矗立着半球形纳米玻璃穹顶，可直视地球悬于地平线附近、永不升落的奇景。'
vibe='广寒市是一座"倒置的城市"——繁华在地下三百米处绽放。熔岩管被改造为层层叠落的环形城区，中央竖井中重力仅及地球六分之一，居民乘坐磁浮胶囊轻盈穿梭，孩童在缓降中翻飞嬉戏。穹顶之下，人造白昼每二十六小时循环一次，暮色时分整城会点亮暖橙色的"思乡灯"，模拟地球黄昏。永恒黑暗的深渊与璀璨的人工星河仅一壁之隔，冷硬的月岩与柔软的水培花园交错，整座城市弥漫着一种孤独而坚韧的前哨气质：每个居民抬头都能看见故乡蓝星，却再也闻不到雨的味道。'
economy='广寒市的经济三大支柱：一、水冰开采与氘燃料精炼——环形山永久阴影区蕴藏的百亿吨水冰被电解为液氢液氧，使这里成为太阳系最大的深空加油站，所有前往火星与木星的船队都必须在此补给；二、氦-3聚变燃料出口——月壤开采的氦-3通过质量投射器射向地球轨道接收站，供养着母星的聚变电网；三、低重力高端制造与科研——近乎真空与微重力环境孕育出地球无法生产的超纯晶体、量子芯片与特种生物制药。此外，"地出 tourism"与低重力养老产业也为城市带来源源不断的信用点。'